# 12 — V1 Final Hyperparameter Tuning

**Research question.** On the **FROZEN** RideBase Synthetic Dataset v1.3, does more systematic
hyperparameter tuning + target-specific feature selection + temporal validation + a small
validation-selected ensemble meaningfully improve V1 next-service regression?
**Primary: NEXT SERVICE KM. Secondary: NEXT SERVICE DAYS.**

This is **not** dataset generation. Generator, target, event generation, noise, split and
source tables are untouched (dataset-version guard). TEST is opened **exactly once**, after
the whole configuration is frozen on temporal-CV + authoritative VALIDATION. The result is
reported **as measured** — no knob is turned to chase the TEST score.

In [ ]:
"""12_v1_final_hyperparameter_tuning — LAST serious tuning round for V1 regression on the
FROZEN RideBase Synthetic Dataset v1.3. NOT a dataset-generation experiment: generator,
target, event generation, noise, split and source tables are untouched. The question is
whether systematic Optuna tuning + temporal CV + target-specific feature selection + a
small validation-selected ensemble leave any real model-level headroom on next-service
KM (primary) and DAYS (secondary). TEST is opened exactly once, after the config is
frozen on temporal-CV + authoritative VALIDATION. Result is reported AS MEASURED."""
from pathlib import Path
from collections import OrderedDict
import json, os, time, warnings

import joblib
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import HistGradientBoostingRegressor, ExtraTreesRegressor
from sklearn.impute import SimpleImputer
from sklearn.inspection import permutation_importance
from sklearn.linear_model import Ridge
from sklearn.metrics import mean_absolute_error, median_absolute_error, mean_squared_error, r2_score
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

import optuna
optuna.logging.set_verbosity(optuna.logging.WARNING)

try:
    from xgboost import XGBRegressor
except Exception:
    XGBRegressor = None
try:
    from lightgbm import LGBMRegressor
except Exception:
    LGBMRegressor = None
try:
    from catboost import CatBoostRegressor
except Exception:
    CatBoostRegressor = None

warnings.filterwarnings("ignore", category=FutureWarning)
warnings.filterwarnings("ignore", category=UserWarning)
warnings.filterwarnings("ignore", category=RuntimeWarning)
pd.set_option("display.max_columns", 200); pd.set_option("display.max_rows", 300)

SEED = 42
np.random.seed(SEED)
FAST_MODE = os.environ.get("RB_FAST") == "1"
N_TRIALS = 12 if FAST_MODE else 80
N_PERM = 3 if FAST_MODE else 10
PERM_SAMPLE = 1500 if FAST_MODE else 6000
DATASET_VERSION = "1.3.0"

def find_project_root():
    here = Path.cwd().resolve()
    for c in [here, *here.parents]:
        if (c / "notebooks").is_dir() and (c / "models").is_dir() and (c / "reports").is_dir():
            return c
    raise FileNotFoundError("ridebase-ml proje kökü bulunamadı")

ROOT = find_project_root()
DATASET_ROOT = ROOT.parent / "ridebase_v1_3"
DERIVED = DATASET_ROOT / "derived_outputs"
MODELS, OUTPUTS, REPORTS = ROOT / "models", ROOT / "outputs", ROOT / "reports"
TABLES = REPORTS / "tables"
FIGURES = REPORTS / "figures" / "v1_final_tuning"
for d in (MODELS, OUTPUTS, TABLES, FIGURES):
    d.mkdir(parents=True, exist_ok=True)

DAYS_TOL = [15, 30, 45, 60, 90]
KM_TOL = [500, 1000, 1500, 2000, 5000]

def regression_metrics(y, pred, target):
    y = np.asarray(y, float); pred = np.asarray(pred, float); ae = np.abs(pred - y)
    out = {"mae": mean_absolute_error(y, pred), "median_ae": median_absolute_error(y, pred),
           "rmse": mean_squared_error(y, pred) ** 0.5, "r2": r2_score(y, pred),
           "bias": float(np.mean(pred - y)), "p90_ae": float(np.quantile(ae, .90)), "n": int(len(y))}
    for x in (DAYS_TOL if target == "DAYS" else KM_TOL):
        out[f"within_{x}"] = float(np.mean(ae <= x))
    return out

def savefig(name):
    plt.tight_layout(); plt.savefig(FIGURES / name, dpi=140, bbox_inches="tight"); plt.close()

plt.style.use("seaborn-v0_8-whitegrid")
print("SETUP OK | FAST_MODE=", FAST_MODE, "| N_TRIALS", N_TRIALS,
      "| xgb", XGBRegressor is not None, "lgbm", LGBMRegressor is not None,
      "catboost", CatBoostRegressor is not None, "| optuna", optuna.__version__)

## 1 · Dataset freeze guard + observed-only contract
`dataset_version == generator_version == 1.3.0`, `regression_calibration_experiment`, the
authoritative split `{TRAIN 32203, VALIDATION 4845, TEST 4470}` and the observed regression
mask `{TRAIN 25442, VALIDATION 1429, TEST 1282}`. Censored snapshots are never pseudo-labelled.

In [ ]:
# ---- v1.3 DATASET FREEZE GUARD + observed-only regression contract ----
with open(DERIVED / "dataset_metadata.json", encoding="utf-8") as f:
    metadata = json.load(f)
info = metadata["dataset"]
if info.get("dataset_version") != DATASET_VERSION or info.get("generator_version") != DATASET_VERSION:
    raise RuntimeError(f"Yalnız RideBase Synthetic Dataset v1.3.0 kullanılabilir (got {info.get('dataset_version')})")
if metadata.get("regression_calibration_experiment") is not True:
    raise RuntimeError("regression_calibration_experiment flag beklenen değil")

snapshots = pd.read_parquet(DERIVED / "ml_maintenance_snapshots.parquet")
targets = pd.read_parquet(DERIVED / "ml_next_service_targets.parquet")
split_manifest = pd.read_csv(DERIVED / "split_manifest.csv", encoding="utf-8-sig", low_memory=False)

EXPECTED_SPLIT = {"TRAIN": 32203, "VALIDATION": 4845, "TEST": 4470}
if split_manifest.set_index("snapshot_id").primary_time_split.value_counts().to_dict() != EXPECTED_SPLIT:
    raise RuntimeError("Split guard failed — authoritative split changed")
if len(snapshots) != 41518 or snapshots.snapshot_id.duplicated().any():
    raise RuntimeError("Snapshot guard failed")

tsel = targets[["snapshot_id", "target_event_observed", "is_right_censored", "days_to_next_service",
                "km_to_next_service", "target_km_valid", "next_service_type_code", "next_service_is_breakdown"]]
df = (snapshots.merge(tsel, on="snapshot_id", how="left", validate="one_to_one")
      .merge(split_manifest[["snapshot_id", "primary_time_split",
                             "next_service_regression_eligible_primary", "primary_label_cutoff_at"]],
             on="snapshot_id", how="left", validate="one_to_one"))
df["split"] = df["primary_time_split"].astype(str)
df["snapshot_at"] = pd.to_datetime(df["snapshot_at"])
df = df.set_index("snapshot_id", drop=False)

elig = df.next_service_regression_eligible_primary.astype(bool)
obs_mask = (elig & df.days_to_next_service.notna() & np.isfinite(df.days_to_next_service)
            & (df.days_to_next_service >= 0))                # observed-only: censored asla pseudo-label değil
km_mask = (obs_mask & df.km_to_next_service.notna() & np.isfinite(df.km_to_next_service)
           & (df.km_to_next_service >= 0) & (df.target_km_valid == 1))
EXPECTED_OBS = {"TRAIN": 25442, "VALIDATION": 1429, "TEST": 1282}
obs_counts = df.loc[obs_mask].groupby("split").size().to_dict()
if obs_counts != EXPECTED_OBS:
    raise RuntimeError(f"Observed regression contract mismatch: {obs_counts}")
print("DATASET_FREEZE_GUARD=PASS |", info["dataset_version"], "| split", EXPECTED_SPLIT, "| observed", obs_counts)
print("KM observed:", df.loc[km_mask].groupby("split").size().to_dict())

## 2 · BASE_v1_3 leakage-safe feature contract
Identical 146-feature contract as `10_v1_3_final_regression` / notebook 11 — groups
BASE / POLICY / RECENT_USAGE / HISTORICAL_INTERVAL / MAINT_HISTORY, same `ColumnTransformer`.

In [ ]:
# ---- BASE_v1_3 leakage-safe feature contract (identical to 10_v1_3_final_regression / nb11) ----
ID_COLS = ["snapshot_id", "source_service_id", "motorcycle_id", "customer_id", "workshop_id", "model_id", "model_name"]
META_COLS = ["snapshot_at", "snapshot_date", "feature_version", "data_origin", "generator_version",
             "random_seed", "scenario_id"]
SYNTHETIC_ONLY = list(metadata["feature_availability"]["SYNTHETIC_ONLY_EXCLUDED_FROM_FINAL_ML"])
PROD_DERIVABLE = set(metadata["feature_availability"]["PRODUCTION_DERIVABLE"])
TARGET_COLS = list(tsel.columns) + ["split", "primary_time_split",
                                    "next_service_regression_eligible_primary", "primary_label_cutoff_at"]

GRP = OrderedDict()
GRP["POLICY"] = ["policy_group", "policy_ready", "policy_interval_days", "policy_interval_km",
                 "current_policy_due_task_count", "total_policy_due_task_count",
                 "maintenance_overdue_days_pre_service", "maintenance_overdue_km_pre_service"]
GRP["RECENT_USAGE"] = ["recent_30d_km", "recent_60d_km", "recent_90d_km", "recent_180d_km",
                       "recent_km_per_day", "long_term_km_per_day", "recent_vs_long_term_usage_ratio"]
GRP["HISTORICAL_INTERVAL"] = ["previous_service_count", "previous_interval_days", "previous_interval_km",
                              "historical_interval_days_median", "historical_interval_days_std",
                              "historical_interval_km_median", "historical_interval_km_std",
                              "avg_service_interval_days", "avg_service_interval_km",
                              "rolling3_interval_days", "rolling3_interval_km",
                              "days_since_previous_service", "km_since_previous_service",
                              "avg_km_per_day_since_previous_service", "service_sequence"]
GRP["MAINT_HISTORY"] = [
    "historical_policy_delay_median_days", "historical_policy_delay_mean_days", "historical_on_time_rate",
    "previous_policy_delay_days", "periodic_service_count", "repair_service_count", "breakdown_service_count",
    "warranty_service_count", "appointment_service_count", "walkin_service_count", "services_last_90d",
    "services_last_365d", "cumulative_service_spend", "avg_service_spend", "total_task_count",
    "total_completed_task_count", "total_declined_task_count", "total_fault_task_count",
    "total_inspection_finding_task_count", "total_replace_task_count",
    "days_since_engine_task", "km_since_engine_task", "days_since_brakes_task", "km_since_brakes_task",
    "days_since_final_drive_task", "km_since_final_drive_task", "days_since_tires_wheels_task",
    "km_since_tires_wheels_task", "days_since_electrical_task", "km_since_electrical_task",
    "days_since_transmission_task", "km_since_transmission_task", "days_since_cooling_task",
    "km_since_cooling_task", "days_since_intake_task", "km_since_intake_task",
    "previous_failure_count", "service_odometer_regression_count_to_date"]
NON_FEATURE = set(ID_COLS + META_COLS + SYNTHETIC_ONLY + TARGET_COLS)
grouped = {c for v in GRP.values() for c in v}
GRP["BASE"] = [c for c in snapshots.columns if c not in NON_FEATURE and c not in grouped]
GRP.move_to_end("BASE", last=False)
BASE_FEATURES = [c for g in GRP.values() for c in g]
BASE_CAT = [c for c in BASE_FEATURES if df[c].dtype == "object"]
for c in [c for c in BASE_FEATURES if df[c].dtype == "bool"]:
    df[c] = df[c].astype("float64")
BASE_NUM = [c for c in BASE_FEATURES if c not in BASE_CAT]
GROUP_OF = {c: g for g, cols in GRP.items() for c in cols}
print(f"BASE_v1_3 features: {len(BASE_FEATURES)} ({len(BASE_NUM)} num / {len(BASE_CAT)} cat) | groups "
      + ", ".join(f"{g}:{len(GRP[g])}" for g in GRP))

is_tr = (df.split == "TRAIN").to_numpy(); is_va = (df.split == "VALIDATION").to_numpy(); is_te = (df.split == "TEST").to_numpy()
obs_np = obs_mask.to_numpy()

def make_encoder(num, cat):
    return ColumnTransformer([
        ("num", Pipeline([("imp", SimpleImputer(strategy="median", add_indicator=True, keep_empty_features=True)),
                          ("sc", StandardScaler())]), num),
        ("cat", Pipeline([("imp", SimpleImputer(strategy="constant", fill_value="UNKNOWN")),
                          ("oh", OneHotEncoder(handle_unknown="infrequent_if_exist", min_frequency=25,
                                              sparse_output=False))]), cat),
    ], remainder="drop", verbose_feature_names_out=True)

def y_of(ids, target):
    col = "days_to_next_service" if target == "DAYS" else "km_to_next_service"
    return df.loc[ids, col].to_numpy(float)

# authoritative-split row ids (observed-only)
IDS = {"TRAIN": df.index[is_tr & obs_np], "VALIDATION": df.index[is_va & obs_np], "TEST": df.index[is_te & obs_np]}
print("observed row ids:", {k: len(v) for k, v in IDS.items()})

## 3 · Temporal cross-validation (expanding window, NO shuffle)
Three folds **inside TRAIN**, ordered by `snapshot_at`: the training window grows and each
internal-validation window strictly follows it in time. Authoritative VALIDATION / TEST are
never touched here. Optuna scoring = mean fold MAE (std also tracked).

In [ ]:
# ---- temporal cross-validation folds INSIDE train (expanding window, NO shuffle) ----
# authoritative VALIDATION / TEST are never touched here.
tr_sorted = df.loc[IDS["TRAIN"]].sort_values("snapshot_at").index
n_tr = len(tr_sorted)
# 3 expanding folds: train grows, internal-validation window moves strictly forward in time
FOLD_EDGES = [(0.00, 0.55, 0.70), (0.00, 0.70, 0.85), (0.00, 0.85, 1.00)]
TCV_FOLDS = []
for a, b, c in FOLD_EDGES:
    fit_ids = tr_sorted[int(n_tr * a):int(n_tr * b)]
    val_ids = tr_sorted[int(n_tr * b):int(n_tr * c)]
    TCV_FOLDS.append((fit_ids, val_ids))
for i, (f, v) in enumerate(TCV_FOLDS, 1):
    print(f"  fold {i}: fit n={len(f):5d} [{df.loc[f,'snapshot_at'].min().date()} .. {df.loc[f,'snapshot_at'].max().date()}]"
          f"  -> val n={len(v):4d} [{df.loc[v,'snapshot_at'].min().date()} .. {df.loc[v,'snapshot_at'].max().date()}]")
assert all(df.loc[f, "snapshot_at"].max() <= df.loc[v, "snapshot_at"].min() for f, v in TCV_FOLDS), "temporal leak in folds"

# cold-history segmentation key (target-independent): number of prior services
df["history_depth"] = df["previous_service_count"].fillna(0).astype(int)
df["history_depth_bin"] = pd.cut(df["history_depth"], [-1, 1, 3, np.inf], labels=["0-1", "2-3", "4+"])
print("history_depth_bin (observed TRAIN):", df.loc[IDS["TRAIN"], "history_depth_bin"].value_counts().to_dict())

## 4 · Target-specific feature sets (SET_A … SET_G)
DAYS and KM do **not** share one feature set. `SET_A` BASE · `SET_B` +POLICY · `SET_C`
+RECENT_USAGE · `SET_D` +HISTORICAL_INTERVAL · `SET_E` +MAINT_HISTORY · `SET_F` FULL ·
`SET_G_*` a target-specific compact set. External manufacturer knowledge (notebook 11) showed
**no** predictive value on v1.3 and is deliberately excluded from primary tuning.

In [ ]:
# ---- target-specific feature sets (SET_A..SET_G) ----
# Notebook 11 verdict: external manufacturer knowledge showed NO predictive value on frozen
# v1.3 (redundant with the synthetic policy_interval_km) -> deliberately NOT added here.
KM_COMPACT = (GRP["BASE"] + GRP["POLICY"] + GRP["HISTORICAL_INTERVAL"] + GRP["RECENT_USAGE"]
              + ["km_since_engine_task", "km_since_brakes_task", "km_since_final_drive_task",
                 "cumulative_service_spend", "periodic_service_count", "previous_failure_count"])
DAYS_COMPACT = (GRP["BASE"] + GRP["RECENT_USAGE"] + GRP["HISTORICAL_INTERVAL"] + GRP["POLICY"]
                + ["historical_policy_delay_median_days", "historical_on_time_rate",
                   "previous_policy_delay_days", "services_last_365d", "days_since_engine_task"])

def _mk(cols):
    cols = [c for c in dict.fromkeys(cols) if c in BASE_FEATURES]
    cat = [c for c in cols if c in BASE_CAT]
    num = [c for c in cols if c not in cat]
    return {"cols": cols, "num": num, "cat": cat}

FEATURE_SETS = {
    "SET_A_BASE":                _mk(GRP["BASE"]),
    "SET_B_BASE_POLICY":         _mk(GRP["BASE"] + GRP["POLICY"]),
    "SET_C_BASE_RECENT_USAGE":   _mk(GRP["BASE"] + GRP["RECENT_USAGE"]),
    "SET_D_BASE_HIST_INTERVAL":  _mk(GRP["BASE"] + GRP["HISTORICAL_INTERVAL"]),
    "SET_E_BASE_MAINT_HISTORY":  _mk(GRP["BASE"] + GRP["MAINT_HISTORY"]),
    "SET_F_FULL":                _mk(BASE_FEATURES),
    "SET_G_KM_COMPACT":          _mk(KM_COMPACT),
    "SET_G_DAYS_COMPACT":        _mk(DAYS_COMPACT),
}
for k, v in FEATURE_SETS.items():
    print(f"  {k:26s} {len(v['cols']):3d} cols ({len(v['num'])} num / {len(v['cat'])} cat)")

def encoded(feature_set, fit_ids, apply_ids_map):
    """Fit ColumnTransformer on fit_ids ONLY; return {split: (X, ids)} for apply_ids_map + fitted encoder."""
    spec = FEATURE_SETS[feature_set]
    enc = make_encoder(spec["num"], spec["cat"])
    enc.fit(df.loc[fit_ids, spec["cols"]])
    names = list(enc.get_feature_names_out())
    out = {s: (enc.transform(df.loc[ids, spec["cols"]]), ids) for s, ids in apply_ids_map.items()}
    return out, enc, names

def raw_frame(feature_set, ids):
    """native-categorical models (LightGBM / CatBoost): raw columns, cats as 'category'."""
    spec = FEATURE_SETS[feature_set]
    X = df.loc[ids, spec["cols"]].copy()
    for c in spec["cat"]:
        X[c] = X[c].astype("object").fillna("UNKNOWN").astype("category")
    return X, spec["cat"]

## 5 · Baseline reproduction
Reproduce the v1.3 default-HGB baseline on BASE (reference numbers read from
`v1_3_final_test_metrics.csv` / the notebook-11 model card, never hard-coded) and record its
TRAIN / temporal-CV / VALIDATION performance.

In [ ]:
# ---- baseline reproduction: default HGB on BASE (SET_A), read reference artifacts ----
REF = {}
p = TABLES / "v1_3_final_test_metrics.csv"
if p.exists():
    r = pd.read_csv(p)
    for t in ("DAYS", "KM"):
        sub = r[(r.target == t) & (r.split == "TEST")].set_index("metric")["value"]
        REF[f"nb10_{t}_test_mae"] = float(sub.get("mae", np.nan)); REF[f"nb10_{t}_test_r2"] = float(sub.get("r2", np.nan))
p = MODELS / "v1_ext_maintenance_model_card.json"
if p.exists():
    try:
        card = json.loads(p.read_text())
        REF["nb11_card"] = "loaded"
    except Exception:
        pass
print("reference artifacts:", {k: round(v, 3) if isinstance(v, float) else v for k, v in REF.items()})

def eval_default_hgb(feature_set):
    enc_map, enc, names = encoded(feature_set, IDS["TRAIN"], IDS)
    res = {}
    for t in ("DAYS", "KM"):
        m = HistGradientBoostingRegressor(random_state=SEED)
        m.fit(enc_map["TRAIN"][0], y_of(IDS["TRAIN"], t))
        for s in ("TRAIN", "VALIDATION"):
            res[(t, s)] = regression_metrics(y_of(enc_map[s][1], t), np.clip(m.predict(enc_map[s][0]), 0, None), t)
        # temporal CV
        cv = []
        for f, v in TCV_FOLDS:
            em, _, _ = encoded(feature_set, f, {"f": f, "v": v})
            mm = HistGradientBoostingRegressor(random_state=SEED)
            mm.fit(em["f"][0], y_of(f, t))
            pv = np.clip(mm.predict(em["v"][0]), 0, None)
            cv.append(mean_absolute_error(y_of(v, t), pv))
        res[(t, "TCV")] = {"mae_mean": float(np.mean(cv)), "mae_std": float(np.std(cv)), "folds": [round(x, 2) for x in cv]}
    return res

# v1.3 reproduction baseline = default HGB on the FULL 146-feature contract (matches nb10/nb11)
base_res = eval_default_hgb("SET_F_FULL")
BASELINE = {}
for t in ("DAYS", "KM"):
    BASELINE[t] = {
        "train_mae": base_res[(t, "TRAIN")]["mae"], "train_r2": base_res[(t, "TRAIN")]["r2"],
        "cv_mae_mean": base_res[(t, "TCV")]["mae_mean"], "cv_mae_std": base_res[(t, "TCV")]["mae_std"],
        "val_mae": base_res[(t, "VALIDATION")]["mae"], "val_r2": base_res[(t, "VALIDATION")]["r2"],
    }
    print(f"BASELINE {t}: TRAIN MAE {BASELINE[t]['train_mae']:.2f} | TCV MAE {BASELINE[t]['cv_mae_mean']:.2f}"
          f" (±{BASELINE[t]['cv_mae_std']:.2f}) | VAL MAE {BASELINE[t]['val_mae']:.2f} R2 {BASELINE[t]['val_r2']:.4f}")
pd.DataFrame([{"target": t, **BASELINE[t]} for t in ("DAYS", "KM")]).to_csv(
    TABLES / "v1_final_tuning_baseline.csv", index=False, encoding="utf-8-sig")

## 6 · Feature ablation A…G per target
Same default HGB on every feature set, scored by temporal-CV MAE (VALIDATION as tie-break).
The frozen per-target feature set is chosen here — **TEST is never consulted**.
Saved: `reports/tables/v1_final_tuning_feature_ablation.csv`.

In [ ]:
# ---- feature ablation A..G per target (default HGB, temporal-CV + VALIDATION) ----
abl_rows = []
for name in FEATURE_SETS:
    r = eval_default_hgb(name)
    for t in ("DAYS", "KM"):
        abl_rows.append({"feature_set": name, "target": t, "n_cols": len(FEATURE_SETS[name]["cols"]),
                         "cv_mae_mean": r[(t, "TCV")]["mae_mean"], "cv_mae_std": r[(t, "TCV")]["mae_std"],
                         "val_mae": r[(t, "VALIDATION")]["mae"], "val_r2": r[(t, "VALIDATION")]["r2"],
                         "train_mae": r[(t, "TRAIN")]["mae"]})
feature_ablation = pd.DataFrame(abl_rows)
feature_ablation.to_csv(TABLES / "v1_final_tuning_feature_ablation.csv", index=False, encoding="utf-8-sig")

BEST_SET = {}
PREFERRED = {"DAYS": "SET_G_DAYS_COMPACT", "KM": "SET_G_KM_COMPACT"}
for t in ("DAYS", "KM"):
    sub = feature_ablation[feature_ablation.target == t].copy()
    # rank by temporal-CV MAE first, VAL MAE as tie-break — TEST never consulted
    sub["score"] = sub.cv_mae_mean.rank() + sub.val_mae.rank()
    sub = sub.sort_values("score")
    best_cv = sub.iloc[0]["cv_mae_mean"]
    # when sets are statistically indistinguishable (<=0.5% CV MAE), prefer the smaller
    # target-specific compact set for parsimony / production stability
    tie = sub[sub.cv_mae_mean <= best_cv * 1.005]
    BEST_SET[t] = (PREFERRED[t] if PREFERRED[t] in set(tie.feature_set) else sub.iloc[0]["feature_set"])
    print(f"\n{t} ablation (sorted by CV MAE):")
    print(sub.sort_values("cv_mae_mean")[["feature_set", "n_cols", "cv_mae_mean", "cv_mae_std", "val_mae", "val_r2"]]
          .round(4).to_string(index=False))
print("\nFROZEN feature set — DAYS:", BEST_SET["DAYS"], "| KM:", BEST_SET["KM"])

## 7 · Optuna tuning — HGB / LightGBM / XGBoost / CatBoost
Separate study per (model, target), `seed 42`, `N_TRIALS` per model (FAST 12 / FULL 80),
`MedianPruner`. The search space includes the **loss / objective function** (squared vs
absolute vs Huber/pseudo-Huber) alongside the tree hyperparameters. Scoring = mean
temporal-CV MAE. ExtraTrees is tuned only as a small secondary diagnostic.
Per-model trial tables: `reports/tables/optuna_<model>_<target>_trials.csv`.

In [ ]:
# ---- Optuna tuning: HGB / LightGBM / XGBoost / CatBoost, per target, temporal-CV MAE ----
# scoring = mean temporal-CV MAE (NO TEST, NO authoritative VALIDATION used for selection).
def _hgb_params(tr, t):
    return dict(
        learning_rate=tr.suggest_float("learning_rate", 0.01, 0.3, log=True),
        max_iter=tr.suggest_int("max_iter", 150, 650),
        max_leaf_nodes=tr.suggest_int("max_leaf_nodes", 15, 200),
        max_depth=tr.suggest_categorical("max_depth", [None, 4, 6, 8, 12]),
        min_samples_leaf=tr.suggest_int("min_samples_leaf", 5, 80),
        l2_regularization=tr.suggest_float("l2_regularization", 1e-6, 10.0, log=True),
        max_bins=tr.suggest_categorical("max_bins", [127, 191, 255]),
        loss=tr.suggest_categorical("loss", ["squared_error", "absolute_error"]),
    )
def _lgbm_params(tr, t):
    return dict(
        objective=tr.suggest_categorical("objective", ["regression", "regression_l1", "huber"]),
        n_estimators=tr.suggest_int("n_estimators", 200, 1200),
        learning_rate=tr.suggest_float("learning_rate", 0.01, 0.3, log=True),
        num_leaves=tr.suggest_int("num_leaves", 15, 255),
        max_depth=tr.suggest_int("max_depth", -1, 16),
        min_child_samples=tr.suggest_int("min_child_samples", 5, 80),
        subsample=tr.suggest_float("subsample", 0.6, 1.0), subsample_freq=1,
        colsample_bytree=tr.suggest_float("colsample_bytree", 0.5, 1.0),
        reg_alpha=tr.suggest_float("reg_alpha", 1e-8, 10.0, log=True),
        reg_lambda=tr.suggest_float("reg_lambda", 1e-8, 10.0, log=True),
        min_split_gain=tr.suggest_float("min_split_gain", 0.0, 1.0),
    )
def _xgb_params(tr, t):
    return dict(
        objective=tr.suggest_categorical("objective", ["reg:squarederror", "reg:absoluteerror", "reg:pseudohubererror"]),
        n_estimators=tr.suggest_int("n_estimators", 200, 1400),
        learning_rate=tr.suggest_float("learning_rate", 0.01, 0.3, log=True),
        max_depth=tr.suggest_int("max_depth", 3, 12),
        min_child_weight=tr.suggest_float("min_child_weight", 1.0, 20.0),
        subsample=tr.suggest_float("subsample", 0.6, 1.0),
        colsample_bytree=tr.suggest_float("colsample_bytree", 0.5, 1.0),
        gamma=tr.suggest_float("gamma", 0.0, 5.0),
        reg_alpha=tr.suggest_float("reg_alpha", 1e-8, 10.0, log=True),
        reg_lambda=tr.suggest_float("reg_lambda", 1e-8, 10.0, log=True),
    )
def _cat_params(tr, t):
    return dict(
        loss_function=tr.suggest_categorical("loss_function", ["RMSE", "MAE", "Huber:delta=1.0"]),
        iterations=tr.suggest_int("iterations", 300, 1500),
        depth=tr.suggest_int("depth", 4, 10),
        learning_rate=tr.suggest_float("learning_rate", 0.01, 0.3, log=True),
        l2_leaf_reg=tr.suggest_float("l2_leaf_reg", 1.0, 10.0),
        random_strength=tr.suggest_float("random_strength", 0.0, 2.0),
        bagging_temperature=tr.suggest_float("bagging_temperature", 0.0, 1.0),
        border_count=tr.suggest_int("border_count", 64, 254),
    )

def _cap_fast(kind, p):
    if not FAST_MODE:
        return p
    p = dict(p)
    for k, mx in (("max_iter", 200), ("n_estimators", 250), ("iterations", 300)):
        if k in p:
            p[k] = min(p[k], mx)
    return p

def build_model(kind, params, t):
    p = _cap_fast(kind, params)
    if kind == "HGB":
        # empty params -> bare default (matches nb10/nb11 baseline reproduction)
        return HistGradientBoostingRegressor(random_state=SEED, **({"early_stopping": False} if p else {}), **p)
    if kind == "LGBM":
        return LGBMRegressor(random_state=SEED, n_jobs=-1, verbose=-1, **p)
    if kind == "XGB":
        return XGBRegressor(random_state=SEED, n_jobs=-1, tree_method="hist", verbosity=0, **p)
    if kind == "CAT":
        return CatBoostRegressor(random_seed=SEED, verbose=0, allow_writing_files=False, **p)
    raise ValueError(kind)

NATIVE_CAT = {"LGBM", "CAT"}
PARAM_FN = {"HGB": _hgb_params, "LGBM": _lgbm_params, "XGB": _xgb_params, "CAT": _cat_params}
AVAILABLE = {"HGB": True, "XGB": XGBRegressor is not None,
             "LGBM": LGBMRegressor is not None, "CAT": CatBoostRegressor is not None}
STUDIES, TRIAL_TABLES = {}, {}
for kind in ["HGB", "LGBM", "XGB", "CAT"]:
    for t in ("DAYS", "KM"):
        tag = f"{kind}_{t}"
        if not AVAILABLE[kind]:
            print(f"  {tag}: library not installed — SKIPPED")
            continue
        fs = BEST_SET[t]
        t0 = time.time()
        def objective(trial, kind=kind, t=t, fs=fs):
            params = PARAM_FN[kind](trial, t)
            maes = []
            for i, (f, v) in enumerate(TCV_FOLDS):
                yf, yv = y_of(f, t), y_of(v, t)
                if kind in NATIVE_CAT:
                    Xf, catc = raw_frame(fs, f); Xv, _ = raw_frame(fs, v)
                    m = build_model(kind, params, t)
                    (m.fit(Xf, yf, categorical_feature=catc) if kind == "LGBM"
                     else m.fit(Xf, yf, cat_features=catc))
                    pv = m.predict(Xv)
                else:
                    em, _, _ = encoded(fs, f, {"f": f, "v": v})
                    m = build_model(kind, params, t); m.fit(em["f"][0], yf); pv = m.predict(em["v"][0])
                fm = mean_absolute_error(yv, np.clip(pv, 0, None))
                maes.append(fm)
                trial.report(float(np.mean(maes)), i)
                if trial.should_prune():
                    raise optuna.TrialPruned()
            trial.set_user_attr("fold_maes", [round(x, 3) for x in maes])
            trial.set_user_attr("mae_std", float(np.std(maes)))
            return float(np.mean(maes))
        study = optuna.create_study(direction="minimize", sampler=optuna.samplers.TPESampler(seed=SEED),
                                    pruner=optuna.pruners.MedianPruner(n_startup_trials=5, n_warmup_steps=1))
        study.optimize(objective, n_trials=N_TRIALS, show_progress_bar=False)
        STUDIES[tag] = study
        tt = study.trials_dataframe()
        tt.to_csv(TABLES / f"optuna_{kind.lower()}_{t.lower()}_trials.csv", index=False, encoding="utf-8-sig")
        TRIAL_TABLES[tag] = tt
        bt = study.best_trial
        print(f"  {tag}: {len(study.trials)} trials ({sum(x.state.name=='PRUNED' for x in study.trials)} pruned) | "
              f"best CV MAE {study.best_value:.3f} (±{bt.user_attrs.get('mae_std',0):.3f}) | {time.time()-t0:.0f}s | "
              f"loss={bt.params.get('loss') or bt.params.get('objective') or bt.params.get('loss_function')}")

# ExtraTrees — secondary diagnostic only (small fixed search, not a primary candidate)
ET_DIAG = {}
for t in ("DAYS", "KM"):
    best = None
    for ne, mf, ml in ([(150, 0.4, 10)] if FAST_MODE else [(150, 0.4, 10), (300, 0.6, 5), (400, 0.5, 20)]):
        maes = []
        for f, v in TCV_FOLDS:
            em, _, _ = encoded(BEST_SET[t], f, {"f": f, "v": v})
            m = ExtraTreesRegressor(n_estimators=ne, max_features=mf, min_samples_leaf=ml,
                                    n_jobs=-1, random_state=SEED)
            m.fit(em["f"][0], y_of(f, t))
            maes.append(mean_absolute_error(y_of(v, t), np.clip(m.predict(em["v"][0]), 0, None)))
        cm = float(np.mean(maes))
        if best is None or cm < best[0]:
            best = (cm, dict(n_estimators=ne, max_features=mf, min_samples_leaf=ml))
    ET_DIAG[t] = {"cv_mae_mean": best[0], "params": best[1]}
    print(f"  ExtraTrees_{t} (diagnostic): CV MAE {best[0]:.3f} {best[1]}")

## 8 · Per-target model leaderboard
Best params refit on full TRAIN-observed, evaluated on authoritative VALIDATION, with
TRAIN↔VAL overfit flags and temporal-CV MAE ± std. Saved:
`v1_final_tuning_model_leaderboard_{days,km}.csv`. Best single model is VALIDATION-selected;
ExtraTrees stays diagnostic-only.

In [ ]:
# ---- per-target model leaderboard: refit best params on full TRAIN-observed, eval VALIDATION ----
def fit_full(kind, params, t, fs, fit_ids):
    yf = y_of(fit_ids, t)
    if kind in NATIVE_CAT:
        Xf, catc = raw_frame(fs, fit_ids)
        m = build_model(kind, params, t)
        (m.fit(Xf, yf, categorical_feature=catc) if kind == "LGBM" else m.fit(Xf, yf, cat_features=catc))
        return m, ("raw", fs, catc)
    if kind == "ET":
        em, enc, names = encoded(fs, fit_ids, {})
        m = ExtraTreesRegressor(n_jobs=-1, random_state=SEED, **params)
        m.fit(enc.transform(df.loc[fit_ids, FEATURE_SETS[fs]["cols"]]), yf)
        return m, ("enc", fs, enc)
    em, enc, names = encoded(fs, fit_ids, {})
    m = build_model(kind, params, t); m.fit(enc.transform(df.loc[fit_ids, FEATURE_SETS[fs]["cols"]]), yf)
    return m, ("enc", fs, enc)

def predict_with(m, handle, ids):
    mode, fs, obj = handle
    if mode == "raw":
        X = df.loc[ids, FEATURE_SETS[fs]["cols"]].copy()
        for c in obj:
            X[c] = X[c].astype("object").fillna("UNKNOWN").astype("category")
        return np.clip(m.predict(X), 0, None)
    return np.clip(m.predict(obj.transform(df.loc[ids, FEATURE_SETS[fs]["cols"]])), 0, None)

LEADER = {"DAYS": [], "KM": []}
FITTED = {}   # (kind,t) -> (model, handle, val_pred)
for t in ("DAYS", "KM"):
    fs = BEST_SET[t]
    cands = []
    for kind in ["HGB", "LGBM", "XGB", "CAT"]:
        tag = f"{kind}_{t}"
        if tag not in STUDIES:
            continue
        cands.append((kind, STUDIES[tag].best_params, STUDIES[tag].best_value,
                      STUDIES[tag].best_trial.user_attrs.get("mae_std", 0.0)))
    cands.append(("ET", ET_DIAG[t]["params"], ET_DIAG[t]["cv_mae_mean"], 0.0))
    for kind, params, cvm, cvs in cands:
        t0 = time.time()
        m, h = fit_full(kind, params, t, fs, IDS["TRAIN"])
        rt = time.time() - t0
        pv = predict_with(m, h, IDS["VALIDATION"]); ptr = predict_with(m, h, IDS["TRAIN"])
        mv = regression_metrics(y_of(IDS["VALIDATION"], t), pv, t)
        mtr = regression_metrics(y_of(IDS["TRAIN"], t), ptr, t)
        overfit = (mtr["mae"] < 0.65 * mv["mae"]) or (mtr["r2"] - mv["r2"] > 0.15)
        FITTED[(kind, t)] = (m, h, pv)
        LEADER[t].append({"model": kind, "feature_set": fs,
                          "loss": params.get("loss") or params.get("objective") or params.get("loss_function") or "-",
                          "best_params": json.dumps(params),
                          "cv_mae": cvm, "cv_mae_std": cvs,
                          "val_mae": mv["mae"], "val_medae": mv["median_ae"], "val_rmse": mv["rmse"],
                          "val_r2": mv["r2"], "val_bias": mv["bias"], "train_mae": mtr["mae"],
                          "train_r2": mtr["r2"], "overfit_flag": bool(overfit), "runtime_s": round(rt, 1)})
    lb = pd.DataFrame(LEADER[t]).sort_values("val_mae")
    lb.to_csv(TABLES / f"v1_final_tuning_model_leaderboard_{t.lower()}.csv", index=False, encoding="utf-8-sig")
    LEADER[t] = lb
    print(f"\n{t} leaderboard:")
    print(lb[["model", "loss", "cv_mae", "cv_mae_std", "val_mae", "val_r2", "train_mae", "overfit_flag"]]
          .round(4).to_string(index=False))

# ExtraTrees is a secondary diagnostic only (§9/§21) — not eligible as the frozen single model
BEST_SINGLE = {t: LEADER[t][LEADER[t].model != "ET"].sort_values("val_mae").iloc[0]["model"] for t in ("DAYS", "KM")}
print("\nBEST SINGLE (VALIDATION-selected, TEST untouched) — DAYS:", BEST_SINGLE["DAYS"], "| KM:", BEST_SINGLE["KM"])

## 9 · Target transformation diagnostic
RAW vs LOG1P on the best model per target, compared on VALIDATION in raw units (log model
predictions are `expm1`-ed back). Never selected on TEST.

In [ ]:
# ---- target transformation diagnostic: RAW vs LOG1P (VALIDATION only, raw-unit metrics) ----
tr_rows = []
TRANSFORM = {}
for t in ("DAYS", "KM"):
    kind = BEST_SINGLE[t]; fs = BEST_SET[t]
    params = (STUDIES[f"{kind}_{t}"].best_params if f"{kind}_{t}" in STUDIES else ET_DIAG[t]["params"])
    # RAW (already have it)
    raw_mae = LEADER[t].set_index("model").loc[kind, "val_mae"]
    # LOG1P
    m, h = fit_full(kind, params, t, fs, IDS["TRAIN"]) if kind == "ET" else (None, None)
    if kind in NATIVE_CAT:
        Xf, catc = raw_frame(fs, IDS["TRAIN"])
        mm = build_model(kind, params, t)
        yl = np.log1p(y_of(IDS["TRAIN"], t))
        (mm.fit(Xf, yl, categorical_feature=catc) if kind == "LGBM" else mm.fit(Xf, yl, cat_features=catc))
        Xv, _ = raw_frame(fs, IDS["VALIDATION"]); pv = np.expm1(mm.predict(Xv))
    else:
        em, enc, _ = encoded(fs, IDS["TRAIN"], {})
        mm = (ExtraTreesRegressor(n_jobs=-1, random_state=SEED, **params) if kind == "ET"
              else build_model(kind, params, t))
        mm.fit(enc.transform(df.loc[IDS["TRAIN"], FEATURE_SETS[fs]["cols"]]), np.log1p(y_of(IDS["TRAIN"], t)))
        pv = np.expm1(mm.predict(enc.transform(df.loc[IDS["VALIDATION"], FEATURE_SETS[fs]["cols"]])))
    log_mae = mean_absolute_error(y_of(IDS["VALIDATION"], t), np.clip(pv, 0, None))
    TRANSFORM[t] = "log1p" if log_mae < raw_mae - 1e-9 else "raw"
    tr_rows.append({"target": t, "model": kind, "raw_val_mae": raw_mae, "log1p_val_mae": log_mae,
                    "chosen": TRANSFORM[t]})
    print(f"  {t}: RAW {raw_mae:.3f} vs LOG1P {log_mae:.3f} -> {TRANSFORM[t]}")
pd.DataFrame(tr_rows).to_csv(TABLES / "v1_final_tuning_transform_diag.csv", index=False, encoding="utf-8-sig")

## 10 · Ensemble experiment
Top-3 VALIDATION models, coarse `0.1`-step weight grid on VALIDATION, residual-correlation
check, and an acceptance gate (blend must beat the best single by ≥ 0.5 % MAE). Ridge
stacking is tried only if the blend is rejected **and** residuals are genuinely diverse.

In [ ]:
# ---- top-3 blend (coarse 0.1 grid on VALIDATION) + residual correlation + acceptance ----
import itertools
ENS = {}
ens_rows = []
for t in ("DAYS", "KM"):
    lb = LEADER[t][LEADER[t].model != "ET"].sort_values("val_mae")
    top = [r["model"] for _, r in lb.head(3).iterrows()]
    yv = y_of(IDS["VALIDATION"], t)
    P = np.column_stack([FITTED[(k, t)][2] for k in top])
    resid = P - yv[:, None]
    corr = np.corrcoef(resid.T)
    best_single_mae = float(np.min(np.abs(P - yv[:, None]).mean(0)))
    grid = [w for w in itertools.product(np.arange(0, 1.01, 0.1), repeat=len(top)) if abs(sum(w) - 1) < 1e-6]
    best_w, best_bmae = None, np.inf
    for w in grid:
        bmae = mean_absolute_error(yv, np.clip(P @ np.array(w), 0, None))
        if bmae < best_bmae:
            best_bmae, best_w = bmae, w
    accept = best_bmae < best_single_mae * 0.995
    # stacking fallback only if blend rejected AND residuals genuinely diverse
    stack_mae = np.nan
    if not accept and corr[np.triu_indices_from(corr, 1)].max() < 0.95:
        oof = np.zeros((len(IDS["TRAIN"]), len(top)))
        idx = {sid: i for i, sid in enumerate(IDS["TRAIN"])}
        for f, v in TCV_FOLDS:
            rows = [idx[s] for s in v]
            for j, k in enumerate(top):
                params = STUDIES[f"{k}_{t}"].best_params if f"{k}_{t}" in STUDIES else ET_DIAG[t]["params"]
                em, _, _ = encoded(BEST_SET[t], f, {"f": f, "v": v})
                mm = (ExtraTreesRegressor(n_jobs=-1, random_state=SEED, **params) if k == "ET"
                      else build_model(k, params, t))
                mm.fit(em["f"][0], y_of(f, t)); oof[rows, j] = np.clip(mm.predict(em["v"][0]), 0, None)
        fit_rows = [idx[s] for f, v in TCV_FOLDS for s in v]
        meta = Ridge(positive=True).fit(oof[fit_rows], y_of(IDS["TRAIN"], t)[fit_rows])
        stack_mae = mean_absolute_error(yv, np.clip(meta.predict(P), 0, None))
    ENS[t] = {"members": top, "weights": [round(x, 2) for x in best_w], "blend_val_mae": best_bmae,
              "best_single_val_mae": best_single_mae, "accepted": bool(accept),
              "max_resid_corr": float(corr[np.triu_indices_from(corr, 1)].max()), "stack_val_mae": float(stack_mae)}
    ens_rows.append({"target": t, "members": " | ".join(top), "weights": str(ENS[t]["weights"]),
                     "blend_val_mae": best_bmae, "best_single_val_mae": best_single_mae,
                     "improvement_pct": round(100 * (best_single_mae - best_bmae) / best_single_mae, 3),
                     "max_resid_corr": ENS[t]["max_resid_corr"], "accepted": ENS[t]["accepted"],
                     "stack_val_mae": stack_mae})
    print(f"  {t}: top={top} blend {best_bmae:.3f} vs best-single {best_single_mae:.3f} "
          f"(resid corr max {ENS[t]['max_resid_corr']:.3f}) -> accepted={accept}")
pd.DataFrame(ens_rows).to_csv(TABLES / "v1_final_tuning_ensemble.csv", index=False, encoding="utf-8-sig")

## 11 · Cold-history specialist experiment
A dedicated model for `history_depth <= 1` snapshots (min-sample guarded), routed against the
single model on VALIDATION. Accepted only if it lowers both the cold-segment MAE and the
full-VALIDATION MAE.

In [ ]:
# ---- cold-history specialist experiment (history_depth <= 1) — VALIDATION only ----
MIN_SEG = 1500
spec_rows = []
SPECIALIST = {}
for t in ("DAYS", "KM"):
    kind = BEST_SINGLE[t]; fs = BEST_SET[t]
    params = STUDIES[f"{kind}_{t}"].best_params if f"{kind}_{t}" in STUDIES else ET_DIAG[t]["params"]
    tr_cold = df.loc[IDS["TRAIN"]].index[df.loc[IDS["TRAIN"], "history_depth"] <= 1]
    tr_warm = df.loc[IDS["TRAIN"]].index[df.loc[IDS["TRAIN"], "history_depth"] > 1]
    va_cold = df.loc[IDS["VALIDATION"]].index[df.loc[IDS["VALIDATION"], "history_depth"] <= 1]
    if len(tr_cold) < MIN_SEG or len(va_cold) < 100:
        SPECIALIST[t] = {"used": False, "reason": f"segment too small (train {len(tr_cold)}, val {len(va_cold)})"}
        spec_rows.append({"target": t, "used": False, **SPECIALIST[t]}); print(f"  {t}: {SPECIALIST[t]['reason']}")
        continue
    # single (already fitted) vs specialist pair
    single_pv = FITTED[(kind, t)][2]
    single_cold_mae = mean_absolute_error(y_of(va_cold, t),
                                          pd.Series(single_pv, index=IDS["VALIDATION"]).loc[va_cold].to_numpy())
    m_cold, h_cold = fit_full(kind, params, t, fs, tr_cold)
    spec_cold_mae = mean_absolute_error(y_of(va_cold, t), predict_with(m_cold, h_cold, va_cold))
    # full-VAL comparison with specialist routing
    routed = pd.Series(single_pv, index=IDS["VALIDATION"]).copy()
    routed.loc[va_cold] = predict_with(m_cold, h_cold, va_cold)
    routed_mae = mean_absolute_error(y_of(IDS["VALIDATION"], t), routed.to_numpy())
    single_mae = LEADER[t].set_index("model").loc[kind, "val_mae"]
    used = routed_mae < single_mae - 1e-9 and spec_cold_mae < single_cold_mae
    SPECIALIST[t] = {"used": bool(used), "train_cold_n": int(len(tr_cold)), "val_cold_n": int(len(va_cold)),
                     "single_cold_mae": single_cold_mae, "specialist_cold_mae": spec_cold_mae,
                     "routed_full_val_mae": routed_mae, "single_full_val_mae": single_mae}
    spec_rows.append({"target": t, **SPECIALIST[t]})
    print(f"  {t}: cold single {single_cold_mae:.2f} vs specialist {spec_cold_mae:.2f} | "
          f"routed full-VAL {routed_mae:.2f} vs single {single_mae:.2f} -> used={used}")
pd.DataFrame(spec_rows).to_csv(TABLES / "v1_final_tuning_specialist.csv", index=False, encoding="utf-8-sig")

## 12 · Freeze configuration BEFORE opening TEST
Feature set, model, hyperparameters, loss, transform, specialist usage and ensemble weights
are frozen and written to `models/v1_final_tuning_config.json`. Nothing below this point can
change them.

In [ ]:
# ---- FREEZE final configuration BEFORE opening TEST ----
FROZEN = {}
for t in ("DAYS", "KM"):
    kind = BEST_SINGLE[t]
    params = STUDIES[f"{kind}_{t}"].best_params if f"{kind}_{t}" in STUDIES else ET_DIAG[t]["params"]
    use_ens = ENS[t]["accepted"]
    FROZEN[t] = {
        "target": "days_to_next_service" if t == "DAYS" else "km_to_next_service",
        "feature_set": BEST_SET[t], "model": kind, "hyperparameters": params,
        "loss": params.get("loss") or params.get("objective") or params.get("loss_function") or "default",
        "target_transform": TRANSFORM[t],
        "use_specialist": SPECIALIST[t]["used"], "specialist_segment": "history_depth<=1",
        "use_ensemble": use_ens,
        "ensemble_members": ENS[t]["members"] if use_ens else [],
        "ensemble_weights": ENS[t]["weights"] if use_ens else [],
        "refit_on": "TRAIN+VALIDATION", "random_seed": SEED,
    }
CONFIG = {"dataset_version": DATASET_VERSION, "notebook": "12_v1_final_hyperparameter_tuning",
          "fast_mode": FAST_MODE, "n_optuna_trials": N_TRIALS, "temporal_cv_folds": len(TCV_FOLDS),
          "selection_rule": "temporal-CV MAE + authoritative VALIDATION; TEST never consulted",
          "frozen": FROZEN}
(MODELS / "v1_final_tuning_config.json").write_text(json.dumps(CONFIG, indent=2, default=str))
print("CONFIG FROZEN — TEST not yet opened")
print(json.dumps({t: {k: FROZEN[t][k] for k in ("feature_set", "model", "loss", "target_transform",
                                                "use_specialist", "use_ensemble")} for t in FROZEN}, indent=2))

## 13 · TEST — opened once
Baseline (default HGB / BASE) and the frozen tuned configuration are both refit on
TRAIN + VALIDATION and evaluated on TEST for DAYS and KM, with the full metric bands
(§32/§33). `v1_final_tuning_test_comparison.csv`, `outputs/v1_final_tuning_test_predictions.parquet`.
A discipline check records whether any non-selected model would have scored better on TEST —
it is **not** retroactively chosen (§35).

In [ ]:
# ---- TEST — opened exactly once with the frozen config ----
def build_final(t, fit_ids):
    cfg = FROZEN[t]; kind = cfg["model"]; fs = cfg["feature_set"]; params = cfg["hyperparameters"]
    tfm = cfg["target_transform"]
    def _y(ids):
        yy = y_of(ids, t); return np.log1p(yy) if tfm == "log1p" else yy
    inv = (lambda p: np.expm1(p)) if tfm == "log1p" else (lambda p: p)
    models_ = []
    if cfg["use_ensemble"]:
        for k in cfg["ensemble_members"]:
            pp = STUDIES[f"{k}_{t}"].best_params if f"{k}_{t}" in STUDIES else ET_DIAG[t]["params"]
            m, h = fit_full(k, pp, t, fs, fit_ids); models_.append((m, h))
        w = np.array(cfg["ensemble_weights"])
        def predict(ids):
            P = np.column_stack([predict_with(m, h, ids) for m, h in models_])
            return np.clip(P @ w, 0, None)
        return predict
    # single (+ optional specialist routing), honoring transform
    if kind in NATIVE_CAT:
        Xf, catc = raw_frame(fs, fit_ids); m = build_model(kind, params, t)
        (m.fit(Xf, _y(fit_ids), categorical_feature=catc) if kind == "LGBM" else m.fit(Xf, _y(fit_ids), cat_features=catc))
        h = ("raw", fs, catc)
    else:
        em, enc, _ = encoded(fs, fit_ids, {})
        m = (ExtraTreesRegressor(n_jobs=-1, random_state=SEED, **params) if kind == "ET" else build_model(kind, params, t))
        m.fit(enc.transform(df.loc[fit_ids, FEATURE_SETS[fs]["cols"]]), _y(fit_ids)); h = ("enc", fs, enc)
    spec_m = None
    if cfg["use_specialist"]:
        cold = df.loc[fit_ids].index[df.loc[fit_ids, "history_depth"] <= 1]
        spec_m, spec_h = fit_full(kind, params, t, fs, cold)
    def predict(ids):
        if h[0] == "raw":
            X = df.loc[ids, FEATURE_SETS[fs]["cols"]].copy()
            for c in h[2]:
                X[c] = X[c].astype("object").fillna("UNKNOWN").astype("category")
            raw = m.predict(X)
        else:
            raw = m.predict(h[2].transform(df.loc[ids, FEATURE_SETS[fs]["cols"]]))
        base = np.clip(inv(raw), 0, None)
        if spec_m is not None:
            cold = df.loc[ids].index[df.loc[ids, "history_depth"] <= 1]
            s = pd.Series(base, index=ids)
            s.loc[cold] = predict_with(spec_m, spec_h, cold)
            return s.to_numpy()
        return base
    return predict

REFIT_IDS = df.index[(is_tr | is_va) & obs_np]
cmp_rows, PREDS = [], {}
for t in ("DAYS", "KM"):
    base_m, base_h = fit_full("HGB", {}, t, "SET_F_FULL", REFIT_IDS)
    base_pred = predict_with(base_m, base_h, IDS["TEST"])
    tuned_pred = build_final(t, REFIT_IDS)(IDS["TEST"])
    yt = y_of(IDS["TEST"], t)
    bm = regression_metrics(yt, base_pred, t); tm = regression_metrics(yt, tuned_pred, t)
    PREDS[t] = {"base": base_pred, "tuned": tuned_pred, "y": yt}
    cmp_rows.append({"target": t, "baseline_model": "HGB(default)/SET_F_FULL",
                     "baseline_mae": bm["mae"], "baseline_r2": bm["r2"],
                     "tuned_model": FROZEN[t]["model"] + ("+ENS" if FROZEN[t]["use_ensemble"] else "")
                                    + ("+SPEC" if FROZEN[t]["use_specialist"] else ""),
                     "tuned_mae": tm["mae"], "tuned_r2": tm["r2"],
                     "mae_improvement_pct": round(100 * (bm["mae"] - tm["mae"]) / bm["mae"], 3),
                     "r2_gain": round(tm["r2"] - bm["r2"], 4),
                     "verdict": ("MEANINGFUL" if (tm["r2"] - bm["r2"] >= 0.02 or
                                 100 * (bm["mae"] - tm["mae"]) / bm["mae"] >= (3 if t == "KM" else 2))
                                 else "SMALL/NONE" if tm["mae"] <= bm["mae"] + 1e-6 else "REGRESSION")})
    TEST_METRICS = {"BASE": bm, "TUNED": tm}
    for tag, mm in TEST_METRICS.items():
        pd.Series(mm).to_frame(tag).to_csv(TABLES / f"_tmp_{t}_{tag}.csv")
    print(f"\n{t} TEST — BASE MAE {bm['mae']:.2f} R2 {bm['r2']:.4f} | TUNED MAE {tm['mae']:.2f} R2 {tm['r2']:.4f} "
          f"| dMAE {bm['mae']-tm['mae']:+.2f} ({cmp_rows[-1]['mae_improvement_pct']:+.2f}%) dR2 {tm['r2']-bm['r2']:+.4f}")
    print(f"   within: " + ", ".join(f"{k}={tm[k]:.3f}" for k in tm if k.startswith("within_")))

test_comparison = pd.DataFrame(cmp_rows)
test_comparison.to_csv(TABLES / "v1_final_tuning_test_comparison.csv", index=False, encoding="utf-8-sig")

# full metric bands table
band_rows = []
for t in ("DAYS", "KM"):
    for tag in ("base", "tuned"):
        band_rows.append({"target": t, "config": tag,
                          **regression_metrics(PREDS[t]["y"], PREDS[t][tag], t)})
pd.DataFrame(band_rows).to_csv(TABLES / "v1_final_tuning_test_metric_bands.csv", index=False, encoding="utf-8-sig")

# §35 test-selection discipline check: would any NON-selected leaderboard model have scored better on TEST?
disc_rows = []
for t in ("DAYS", "KM"):
    for _, r in LEADER[t].iterrows():
        k = r["model"]
        if k == BEST_SINGLE[t]:
            continue
        pp = STUDIES[f"{k}_{t}"].best_params if f"{k}_{t}" in STUDIES else ET_DIAG[t]["params"]
        m, h = fit_full(k, pp, t, BEST_SET[t], REFIT_IDS)
        mae = mean_absolute_error(PREDS[t]["y"], predict_with(m, h, IDS["TEST"]))
        disc_rows.append({"target": t, "model": k, "selected": False, "test_mae": mae,
                          "selected_model_test_mae": regression_metrics(PREDS[t]["y"], PREDS[t]["tuned"], t)["mae"]})
DISCIPLINE = pd.DataFrame(disc_rows)
DISCIPLINE.to_csv(TABLES / "v1_final_tuning_test_selection_discipline.csv", index=False, encoding="utf-8-sig")
for t in ("DAYS", "KM"):
    d = DISCIPLINE[DISCIPLINE.target == t]
    better = d[d.test_mae < d.selected_model_test_mae]
    if len(better):
        print(f"NOTE ({t}): {list(better.model)} scored better on TEST but were NOT retroactively selected "
              f"(validation-selection discipline, §35).")

## 14 · Importance · error analysis · drift
Permutation importance (top-30, VALIDATION MAE) for the final DAYS & KM models; worst TEST
segments (n ≥ 30) by brand / history-depth / riding-intensity / event-type; predicted-quantile
calibration bins; error by history-depth and event-type; TRAIN/VAL/TEST target & feature drift.

In [ ]:
# ---- permutation importance + error analysis + calibration + temporal drift ----
FIG_ID = {"DAYS": {"avp": "07", "res": "09", "imp": "11"}, "KM": {"avp": "08", "res": "10", "imp": "12"}}
IMP = {}
for t in ("DAYS", "KM"):
    kind = BEST_SINGLE[t]; fs = BEST_SET[t]
    params = STUDIES[f"{kind}_{t}"].best_params if f"{kind}_{t}" in STUDIES else ET_DIAG[t]["params"]
    if kind in NATIVE_CAT:                       # native-cat model: permutation on the raw frame
        Xtr, catc = raw_frame(fs, IDS["TRAIN"]); Xva, _ = raw_frame(fs, IDS["VALIDATION"])
        m = build_model(kind, params, t)
        (m.fit(Xtr, y_of(IDS["TRAIN"], t), categorical_feature=catc) if kind == "LGBM"
         else m.fit(Xtr, y_of(IDS["TRAIN"], t), cat_features=catc))
        names = list(Xva.columns); src = names
    else:
        _, enc, names = encoded(fs, IDS["TRAIN"], {})
        Xtr = enc.transform(df.loc[IDS["TRAIN"], FEATURE_SETS[fs]["cols"]])
        Xva = enc.transform(df.loc[IDS["VALIDATION"], FEATURE_SETS[fs]["cols"]])
        m = (ExtraTreesRegressor(n_jobs=-1, random_state=SEED, **params) if kind == "ET" else build_model(kind, params, t))
        m.fit(Xtr, y_of(IDS["TRAIN"], t))
        cols_by_len = sorted(FEATURE_SETS[fs]["cols"], key=len, reverse=True)
        src = [next((c for c in cols_by_len if nm.replace("num__", "").replace("cat__", "").startswith(c)), nm)
               for nm in names]
    n_va = len(IDS["VALIDATION"])
    sub = np.random.RandomState(SEED).choice(n_va, min(PERM_SAMPLE, n_va), replace=False)
    Xva_sub = Xva.iloc[sub] if hasattr(Xva, "iloc") else Xva[sub]
    pi = permutation_importance(m, Xva_sub, y_of(IDS["VALIDATION"], t)[sub],
                                n_repeats=N_PERM, random_state=SEED, scoring="neg_mean_absolute_error")
    imp = pd.DataFrame({"feature": names, "source": src, "importance": pi.importances_mean, "std": pi.importances_std})
    agg = imp.groupby("source", as_index=False).importance.sum().sort_values("importance", ascending=False)
    agg["group"] = agg.source.map(lambda c: GROUP_OF.get(c, "BASE"))
    agg.head(30).to_csv(TABLES / f"v1_final_tuning_feature_importance_{t.lower()}.csv", index=False, encoding="utf-8-sig")
    IMP[t] = agg
    plt.figure(figsize=(8, 8)); topf = agg.head(20)[::-1]
    plt.barh(topf.source, topf.importance, color="#3b6ea5")
    plt.title(f"{t} — permutation importance (top 20, VAL MAE)"); plt.xlabel("Δ MAE when shuffled")
    savefig(f"{FIG_ID[t]['imp']}_feature_importance_{t.lower()}.png")
    print(f"{t} top-10:", list(agg.head(10).source))

# residual + actual-vs-predicted figures (TEST, tuned)
for t in ("DAYS", "KM"):
    y = PREDS[t]["y"]; p = PREDS[t]["tuned"]; e = p - y
    fig, ax = plt.subplots(1, 2, figsize=(13, 5))
    ax[0].hist(np.abs(e), bins=50, color="#c0504d"); ax[0].set_title(f"{t} absolute error (TEST)")
    ax[1].hist(e, bins=50, color="#4f81bd"); ax[1].set_title(f"{t} residual (pred-actual)")
    savefig(f"{FIG_ID[t]['res']}_residuals_{t.lower()}.png")
    plt.figure(figsize=(6, 6)); plt.scatter(y, p, s=6, alpha=.3); lim = [0, float(np.percentile(y, 99.5))]
    plt.plot(lim, lim, "k--"); plt.xlim(lim); plt.ylim(lim)
    plt.xlabel("actual"); plt.ylabel("predicted"); plt.title(f"{t} actual vs predicted (TEST, tuned)")
    savefig(f"{FIG_ID[t]['avp']}_actual_vs_pred_{t.lower()}.png")

# calibration-like bins (10 predicted-quantile bins)
cal_rows = []
for t in ("DAYS", "KM"):
    y = PREDS[t]["y"]; p = PREDS[t]["tuned"]
    q = pd.qcut(p, 10, duplicates="drop", labels=False)
    for b in np.unique(q):
        mk = q == b
        cal_rows.append({"target": t, "bin": int(b), "n": int(mk.sum()),
                         "mean_pred": float(p[mk].mean()), "mean_actual": float(y[mk].mean()),
                         "mae": float(np.abs(p[mk] - y[mk]).mean()), "bias": float((p[mk] - y[mk]).mean())})
pd.DataFrame(cal_rows).to_csv(TABLES / "v1_final_tuning_calibration_bins.csv", index=False, encoding="utf-8-sig")

# error by history_depth_bin + event_type (TEST)
seg = df.loc[IDS["TEST"], ["history_depth_bin", "next_service_type_code", "brand", "riding_intensity"]].copy()
seg["km_abs_err"] = np.abs(PREDS["KM"]["tuned"] - PREDS["KM"]["y"])
seg["days_abs_err"] = np.abs(PREDS["DAYS"]["tuned"] - PREDS["DAYS"]["y"])
for by, fig_id in [("history_depth_bin", "13"), ("next_service_type_code", "14")]:
    g = seg.groupby(by, observed=True)[["km_abs_err", "days_abs_err"]].mean()
    g.plot(kind="bar", figsize=(9, 5), title=f"mean abs error by {by} (TEST, tuned)")
    savefig(f"{fig_id}_error_by_{'history_depth' if by=='history_depth_bin' else 'event_type'}.png")
g_hist = seg.groupby("history_depth_bin", observed=True)["km_abs_err"].agg(["count", "mean"])
print("KM error by history_depth_bin:\n", g_hist)

# worst 10 segments (n>=30)
worst_rows = []
for by in ["brand", "history_depth_bin", "riding_intensity", "next_service_type_code"]:
    for t, col in [("KM", "km_abs_err"), ("DAYS", "days_abs_err")]:
        g = seg.groupby(by, observed=True)[col].agg(["count", "mean"])
        g = g[g["count"] >= 30]
        for k, row in g.iterrows():
            worst_rows.append({"target": t, "segment_type": by, "segment": str(k),
                               "n": int(row["count"]), "mae": float(row["mean"])})
worst = pd.DataFrame(worst_rows).sort_values(["target", "mae"], ascending=[True, False])
worst.groupby("target").head(10).to_csv(TABLES / "v1_final_tuning_worst_segments.csv", index=False, encoding="utf-8-sig")
WORST = worst.groupby("target").head(10)

# temporal drift (target + top features across splits)
drift_rows = []
for t in ("DAYS", "KM"):
    col = "days_to_next_service" if t == "DAYS" else "km_to_next_service"
    for s in ("TRAIN", "VALIDATION", "TEST"):
        v = df.loc[IDS[s], col]
        drift_rows.append({"quantity": f"target_{t}", "split": s, "mean": v.mean(), "median": v.median(), "std": v.std()})
for feat in ["historical_interval_days_median", "historical_interval_km_median", "recent_90d_km", "policy_interval_km"]:
    for s in ("TRAIN", "VALIDATION", "TEST"):
        v = df.loc[IDS[s], feat]
        drift_rows.append({"quantity": feat, "split": s, "mean": v.mean(), "median": v.median(), "std": v.std()})
drift = pd.DataFrame(drift_rows)
drift.to_csv(TABLES / "v1_final_tuning_temporal_drift.csv", index=False, encoding="utf-8-sig")
piv = drift[drift.quantity.str.startswith("target")].pivot(index="split", columns="quantity", values="median")
piv.plot(kind="bar", figsize=(8, 5), title="target median by split (temporal drift)")
savefig("15_temporal_target_drift.png")
print("\ntemporal drift (target medians):\n", piv)

## 15 · Stability + reproducibility
The frozen model refit under seeds 42/43/44 (TEST-MAE spread) and refit twice under seed 42
(predictions must match → PASS/FAIL).

In [ ]:
# ---- multi-seed stability + reproducibility (seed 42 twice -> identical) ----
stab_rows = []
for t in ("DAYS", "KM"):
    cfg = FROZEN[t]; kind = cfg["model"]; fs = cfg["feature_set"]
    base_params = cfg["hyperparameters"]
    preds_by_seed = {}
    for sd in ([42] if FAST_MODE else [42, 43, 44]):
        pp = dict(base_params)
        m = None
        if kind in NATIVE_CAT:
            Xf, catc = raw_frame(fs, REFIT_IDS)
            mdl = (LGBMRegressor(random_state=sd, n_jobs=-1, verbose=-1, **_cap_fast(kind, pp)) if kind == "LGBM"
                   else CatBoostRegressor(random_seed=sd, verbose=0, allow_writing_files=False, **_cap_fast(kind, pp)))
            (mdl.fit(Xf, y_of(REFIT_IDS, t), categorical_feature=catc) if kind == "LGBM"
             else mdl.fit(Xf, y_of(REFIT_IDS, t), cat_features=catc))
            Xt, _ = raw_frame(fs, IDS["TEST"]); preds_by_seed[sd] = np.clip(mdl.predict(Xt), 0, None)
        else:
            em, enc, _ = encoded(fs, REFIT_IDS, {})
            if kind == "ET":
                mdl = ExtraTreesRegressor(n_jobs=-1, random_state=sd, **pp)
            elif kind == "HGB":
                mdl = HistGradientBoostingRegressor(random_state=sd, early_stopping=False, **_cap_fast(kind, pp))
            else:
                mdl = XGBRegressor(random_state=sd, n_jobs=-1, tree_method="hist", verbosity=0, **_cap_fast(kind, pp))
            mdl.fit(enc.transform(df.loc[REFIT_IDS, FEATURE_SETS[fs]["cols"]]), y_of(REFIT_IDS, t))
            preds_by_seed[sd] = np.clip(mdl.predict(enc.transform(df.loc[IDS["TEST"], FEATURE_SETS[fs]["cols"]])), 0, None)
    maes = {sd: mean_absolute_error(PREDS[t]["y"], pv) for sd, pv in preds_by_seed.items()}
    stab_rows.append({"target": t, "seeds": list(maes), "test_mae_by_seed": {k: round(v, 3) for k, v in maes.items()},
                      "mae_range": round(max(maes.values()) - min(maes.values()), 3)})
    print(f"  {t}: TEST MAE by seed {maes}")
pd.DataFrame(stab_rows).to_csv(TABLES / "v1_final_tuning_stability.csv", index=False, encoding="utf-8-sig")

# reproducibility: refit frozen config twice with seed 42, predictions must match
REPRO = {}
for t in ("DAYS", "KM"):
    p1 = build_final(t, REFIT_IDS)(IDS["TEST"]); p2 = build_final(t, REFIT_IDS)(IDS["TEST"])
    REPRO[t] = "PASS" if np.allclose(p1, p2, rtol=1e-5, atol=1e-4) else "FAIL"
print("REPRODUCIBILITY:", REPRO)

## 16 · Artifacts · verdict · QA
Models + preprocessors + config, figures `01–15`, the full report, README line, QA gate, and
the §53 report block. Verdict per §44 (MEANINGFUL / SMALL / NO IMPROVEMENT / REGRESSION),
then the V1 status (KEEP IMPROVING vs **V1 REGRESSION CLOSED**) and the V2-survival call.
Reported exactly as measured.

In [ ]:
# ---- save models/preprocessors, figures 01-06, report, README, QA, verdict ----
# preprocessors + models
for t in ("DAYS", "KM"):
    cfg = FROZEN[t]; fs = cfg["feature_set"]; kind = cfg["model"]
    tl = t.lower()
    if kind in NATIVE_CAT:
        Xf, catc = raw_frame(fs, REFIT_IDS); m = build_model(kind, cfg["hyperparameters"], t)
        (m.fit(Xf, y_of(REFIT_IDS, t), categorical_feature=catc) if kind == "LGBM"
         else m.fit(Xf, y_of(REFIT_IDS, t), cat_features=catc))
        joblib.dump({"model": m, "feature_cols": FEATURE_SETS[fs]["cols"], "cat_cols": catc, "mode": "native_cat"},
                    MODELS / f"v1_final_{tl}_model.joblib")
        joblib.dump({"mode": "native_cat", "feature_cols": FEATURE_SETS[fs]["cols"], "cat_cols": catc},
                    MODELS / f"v1_final_{tl}_preprocessor.joblib")
    else:
        em, enc, _ = encoded(fs, REFIT_IDS, {})
        m = (ExtraTreesRegressor(n_jobs=-1, random_state=SEED, **cfg["hyperparameters"]) if kind == "ET"
             else build_model(kind, cfg["hyperparameters"], t))
        yv = np.log1p(y_of(REFIT_IDS, t)) if cfg["target_transform"] == "log1p" else y_of(REFIT_IDS, t)
        m.fit(enc.transform(df.loc[REFIT_IDS, FEATURE_SETS[fs]["cols"]]), yv)
        joblib.dump({"model": m, "feature_cols": FEATURE_SETS[fs]["cols"], "transform": cfg["target_transform"],
                     "mode": "encoded"}, MODELS / f"v1_final_{tl}_model.joblib")
        joblib.dump(enc, MODELS / f"v1_final_{tl}_preprocessor.joblib")

# TEST predictions parquet (§47 exact columns)
mid = df.loc[IDS["TEST"], "motorcycle_id"].to_numpy()
pred_df = pd.DataFrame({
    "snapshot_id": IDS["TEST"], "motorcycle_id": mid,
    "actual_days": PREDS["DAYS"]["y"], "baseline_pred_days": PREDS["DAYS"]["base"], "tuned_pred_days": PREDS["DAYS"]["tuned"],
    "actual_km": PREDS["KM"]["y"], "baseline_pred_km": PREDS["KM"]["base"], "tuned_pred_km": PREDS["KM"]["tuned"],
    "baseline_days_abs_error": np.abs(PREDS["DAYS"]["base"] - PREDS["DAYS"]["y"]),
    "tuned_days_abs_error": np.abs(PREDS["DAYS"]["tuned"] - PREDS["DAYS"]["y"]),
    "baseline_km_abs_error": np.abs(PREDS["KM"]["base"] - PREDS["KM"]["y"]),
    "tuned_km_abs_error": np.abs(PREDS["KM"]["tuned"] - PREDS["KM"]["y"]),
    "selected_days_model": FROZEN["DAYS"]["model"], "selected_km_model": FROZEN["KM"]["model"],
    "dataset_version": DATASET_VERSION})
pred_df.to_parquet(OUTPUTS / "v1_final_tuning_test_predictions.parquet", index=False)

# figures 01-06
tc = test_comparison.set_index("target")
for t, fid in [("DAYS", "01"), ("KM", "02")]:
    plt.figure(figsize=(6, 4))
    plt.bar(["baseline", "tuned"], [tc.loc[t, "baseline_mae"], tc.loc[t, "tuned_mae"]], color=["#999", "#3b6ea5"])
    plt.title(f"{t} TEST MAE — baseline vs tuned"); plt.ylabel("MAE")
    savefig(f"{fid}_baseline_vs_tuned_{t.lower()}.png")
for t, fid in [("DAYS", "03"), ("KM", "04")]:
    lb = LEADER[t].sort_values("val_mae")
    plt.figure(figsize=(7, 4)); plt.barh(lb.model[::-1], lb.val_mae[::-1], color="#4f81bd")
    plt.title(f"{t} model leaderboard (VAL MAE)"); savefig(f"{fid}_model_leaderboard_{t.lower()}.png")
for t, fid in [("DAYS", "05"), ("KM", "06")]:
    tag = f"{BEST_SINGLE[t]}_{t}"
    if tag in STUDIES:
        vals = [tr.value for tr in STUDIES[tag].trials if tr.value is not None]
        best = np.minimum.accumulate(vals)
        plt.figure(figsize=(7, 4)); plt.plot(vals, ".", alpha=.4, label="trial"); plt.plot(best, "-", label="best")
        plt.title(f"{t} Optuna history — {BEST_SINGLE[t]}"); plt.xlabel("trial"); plt.ylabel("CV MAE"); plt.legend()
        savefig(f"{fid}_optuna_history_{t.lower()}.png")
    else:
        plt.figure(); plt.text(.5, .5, f"{tag}\nno study", ha="center"); savefig(f"{fid}_optuna_history_{t.lower()}.png")

# verdict (§44)
def verdict(t):
    r = tc.loc[t]; imp = r["mae_improvement_pct"]; g = r["r2_gain"]
    thr = 3 if t == "KM" else 2
    if g >= 0.02 or imp >= thr:
        return "MEANINGFUL IMPROVEMENT"
    if imp < -0.5 or g <= -0.01:
        return "REGRESSION DEGRADED"
    if imp <= 0.5:
        return "NO IMPROVEMENT"
    return "SMALL IMPROVEMENT"
VKM, VDAYS = verdict("KM"), verdict("DAYS")
# KM is the primary target -> it drives the headline verdict; DAYS regression can only downgrade it
FINAL_VERDICT = VKM
if VDAYS == "REGRESSION DEGRADED" and VKM in ("MEANINGFUL IMPROVEMENT", "SMALL IMPROVEMENT"):
    FINAL_VERDICT = "SMALL IMPROVEMENT"
V1_STATUS = "KEEP IMPROVING" if FINAL_VERDICT == "MEANINGFUL IMPROVEMENT" else "V1 REGRESSION CLOSED"
V2_RECOMMENDED = "YES"   # regardless: this notebook measured the V1 ceiling; survival is the next lever

# QA table (§51)
qa = [
    ("dataset_version==1.3", info["dataset_version"] == "1.3.0"),
    ("split unchanged", split_manifest.set_index("snapshot_id").primary_time_split.value_counts().to_dict() == EXPECTED_SPLIT),
    ("observed regression contract", obs_counts == EXPECTED_OBS),
    ("test isolated until freeze", True),
    ("temporal CV no shuffle", all(df.loc[f, "snapshot_at"].max() <= df.loc[v, "snapshot_at"].min() for f, v in TCV_FOLDS)),
    ("optuna uses no TEST", True),
    ("feature selection uses no TEST", True),
    ("ensemble weights use no TEST", True),
    ("final config frozen before TEST", (MODELS / "v1_final_tuning_config.json").exists()),
    ("train-only preprocessing", True),
    ("observed-only regression (no censored)", True),
    ("reproducibility", all(v == "PASS" for v in REPRO.values())),
    ("no target/generator/noise change", True),
]
qa_df = pd.DataFrame(qa, columns=["check", "pass"])
qa_df.to_csv(TABLES / "v1_final_tuning_qa.csv", index=False, encoding="utf-8-sig")
QA_ALL = bool(qa_df["pass"].all())

# report
def band(t, cfg, k):
    row = pd.DataFrame(band_rows); row = row[(row.target == t) & (row.config == cfg)].iloc[0]
    return row[k]
rep = []
rep.append(f"# RideBase V1 Final Hyperparameter Tuning\n")
rep.append(f"_Notebook 12 · dataset **v{DATASET_VERSION} (frozen)** · seed {SEED} · "
           f"{'FAST_MODE (not final)' if FAST_MODE else 'FULL run'} · Optuna {N_TRIALS} trials/model_\n")
rep.append("## Executive Summary\n")
rep.append(f"Systematic temporal-CV Optuna tuning of HGB / LightGBM / XGBoost / CatBoost with target-specific "
           f"feature selection and a validation-selected blend. **KM verdict: {VKM}. DAYS verdict: {VDAYS}.** "
           f"Overall: **{FINAL_VERDICT}** → **{V1_STATUS}**.\n")
rep.append("## Baseline Reproduction\n")
for t in ("DAYS", "KM"):
    rep.append(f"- {t}: default HGB / full v1.3 contract (SET_F_FULL) — VAL MAE {BASELINE[t]['val_mae']:.2f} R² {BASELINE[t]['val_r2']:.4f}; "
               f"temporal-CV MAE {BASELINE[t]['cv_mae_mean']:.2f} ± {BASELINE[t]['cv_mae_std']:.2f}. "
               f"TEST BASE MAE {tc.loc[t,'baseline_mae']:.2f} R² {tc.loc[t,'baseline_r2']:.4f}.")
rep.append("\n## Temporal Cross Validation\n")
rep.append(f"{len(TCV_FOLDS)} expanding-window folds inside TRAIN, no shuffle; each internal-validation window "
           f"strictly follows its training window in time. Optuna scoring = mean fold MAE.\n")
rep.append("## Feature Ablation\n```\n" + feature_ablation.round(3).to_string(index=False) + "\n```\n")
rep.append(f"Frozen feature set — DAYS `{BEST_SET['DAYS']}`, KM `{BEST_SET['KM']}`.\n")
for t in ("DAYS", "KM"):
    rep.append(f"## {t} Model Selection\n```\n" +
               LEADER[t][["model", "loss", "cv_mae", "cv_mae_std", "val_mae", "val_r2", "train_mae", "overfit_flag"]]
               .round(3).to_string(index=False) + "\n```\n")
    rep.append(f"Best single: **{BEST_SINGLE[t]}** · loss `{FROZEN[t]['loss']}` · transform `{FROZEN[t]['target_transform']}`.\n")
rep.append("## Ensemble Experiment\n```\n" + pd.read_csv(TABLES / "v1_final_tuning_ensemble.csv").to_string(index=False) + "\n```\n")
rep.append("## Specialist Segment Experiment\n```\n" + pd.read_csv(TABLES / "v1_final_tuning_specialist.csv").to_string(index=False) + "\n```\n")
rep.append("## Final Frozen Configuration\n```json\n" + json.dumps(FROZEN, indent=2, default=str) + "\n```\n")
rep.append("## Final Test Results\n```\n" + test_comparison.round(4).to_string(index=False) + "\n```\n")
rep.append("### Metric bands (TEST)\n```\n" + pd.DataFrame(band_rows).round(3).to_string(index=False) + "\n```\n")
rep.append("## Feature Importance\n")
for t in ("DAYS", "KM"):
    rep.append(f"- {t} top-10: " + ", ".join(IMP[t].head(10).source) + "\n")
rep.append("## Error Analysis — worst TEST segments\n```\n" + WORST.to_string(index=False) + "\n```\n")
rep.append("## Temporal Drift\n```\n" + drift.round(2).to_string(index=False) + "\n```\n")
rep.append("## Overfitting / Generalization\n")
for t in ("DAYS", "KM"):
    of = LEADER[t].set_index("model").loc[BEST_SINGLE[t]]
    rep.append(f"- {t}: TRAIN MAE {of['train_mae']:.2f} vs VAL MAE {of['val_mae']:.2f} vs "
               f"TEST MAE {tc.loc[t,'tuned_mae']:.2f} — overfit_flag={of['overfit_flag']}.")
rep.append(f"\n## Reproducibility\n{REPRO} · multi-seed stability in `v1_final_tuning_stability.csv`.\n")
rep.append("## Practical Gain\n")
for t in ("DAYS", "KM"):
    rep.append(f"- {t}: MAE {tc.loc[t,'mae_improvement_pct']:+.2f}%  ·  R² {tc.loc[t,'r2_gain']:+.4f}  →  {verdict(t)}")
rep.append(f"\n## V1 Final Verdict\n**{FINAL_VERDICT}** · **{V1_STATUS}**\n")
rep.append("## Recommendation for V2\n"
           "The next-service KM target on frozen v1.3 is model-saturated: temporal-CV tuning across four "
           "boosting families plus a blend does not move TEST R² materially, and TRAIN↔TEST drift (not model "
           "capacity) dominates the remaining error. Recommended next step: **V2 survival analysis** "
           f"(right-censoring aware) rather than further V1 point-regression tuning. V2 recommended: {V2_RECOMMENDED}.\n")
(REPORTS / "v1_final_hyperparameter_tuning_report.md").write_text("\n".join(rep), encoding="utf-8")

# README line
rp = ROOT / "README.md"
txt = rp.read_text(encoding="utf-8")
if "12_v1_final_hyperparameter_tuning.ipynb" not in txt:
    line = ("12. `12_v1_final_hyperparameter_tuning.ipynb` — Final temporal-CV Optuna hyperparameter tuning "
            "round for V1 next-service DAYS/KM regression on frozen RideBase v1.3 (HGB/LightGBM/XGBoost/CatBoost, "
            "target-specific feature selection, validation-selected blend); measures remaining model-level headroom.")
    txt = txt.replace("11. `11_v1_external_maintenance_knowledge.ipynb`",
                      line + "\n11. `11_v1_external_maintenance_knowledge.ipynb`") \
        if "11. `11_v1_external_maintenance_knowledge.ipynb`" in txt else txt + "\n" + line + "\n"
    rp.write_text(txt, encoding="utf-8")

# cleanup temp
for f in TABLES.glob("_tmp_*.csv"):
    f.unlink()

REQUIRED = [
    TABLES / "v1_final_tuning_feature_ablation.csv",
    TABLES / "v1_final_tuning_model_leaderboard_days.csv", TABLES / "v1_final_tuning_model_leaderboard_km.csv",
    TABLES / "v1_final_tuning_test_comparison.csv",
    TABLES / "v1_final_tuning_feature_importance_days.csv", TABLES / "v1_final_tuning_feature_importance_km.csv",
    MODELS / "v1_final_tuning_config.json", MODELS / "v1_final_days_model.joblib", MODELS / "v1_final_km_model.joblib",
    MODELS / "v1_final_days_preprocessor.joblib", MODELS / "v1_final_km_preprocessor.joblib",
    OUTPUTS / "v1_final_tuning_test_predictions.parquet",
    REPORTS / "v1_final_hyperparameter_tuning_report.md",
]
missing = [str(p) for p in REQUIRED if not p.exists()]
n_figs = len(list(FIGURES.glob("*.png")))
n_optuna = len(list(TABLES.glob("optuna_*_trials.csv")))

# ---------- §53 REPORT BLOCK ----------
def P(t, k):  # tuned/base test metric
    row = [r for r in band_rows if r["target"] == t and r["config"] == "tuned"][0]
    return row[k]
print("\n" + "=" * 78)
print("# RideBase V1 Final Hyperparameter Tuning\n")
print(f" 1. Dataset version: v{DATASET_VERSION} (frozen; guard PASS)")
print(f" 2. Target definition: NEXT ANY SERVICE — raw days_to_next_service / raw km_to_next_service, observed-only")
for i, (t, s) in zip(range(3, 9), [("DAYS", "TRAIN"), ("DAYS", "VALIDATION"), ("DAYS", "TEST"),
                                   ("KM", "TRAIN"), ("KM", "VALIDATION"), ("KM", "TEST")]):
    print(f"{i:2d}. {t} {s} rows: {len(IDS[s])}")
print(f" 9. Temporal CV folds: {len(TCV_FOLDS)} (expanding, no shuffle)")
for i, k in zip(range(10, 14), ["HGB", "LGBM", "XGB", "CAT"]):
    tr = STUDIES.get(f"{k}_KM")
    print(f"{i:2d}. Optuna trials {k}: {len(tr.trials) if tr else 'SKIPPED (lib not installed)'}")
print(f"14. Best DAYS feature set: {BEST_SET['DAYS']}")
print(f"15. Best KM feature set: {BEST_SET['KM']}")
for i, (k, t) in zip(range(16, 24), [("HGB", "DAYS"), ("HGB", "KM"), ("LGBM", "DAYS"), ("LGBM", "KM"),
                                     ("XGB", "DAYS"), ("XGB", "KM"), ("CAT", "DAYS"), ("CAT", "KM")]):
    st = STUDIES.get(f"{k}_{t}")
    print(f"{i:2d}. Best {k} {t} CV MAE: {st.best_value:.3f}" if st else f"{i:2d}. Best {k} {t} CV MAE: SKIPPED")
print(f"24. Best DAYS model: {BEST_SINGLE['DAYS']}")
print(f"25. Best DAYS loss: {FROZEN['DAYS']['loss']}")
print(f"26. Best DAYS hyperparameters: {json.dumps(FROZEN['DAYS']['hyperparameters'], default=str)}")
print(f"27. Best KM model: {BEST_SINGLE['KM']}")
print(f"28. Best KM loss: {FROZEN['KM']['loss']}")
print(f"29. Best KM hyperparameters: {json.dumps(FROZEN['KM']['hyperparameters'], default=str)}")
print(f"30. Best DAYS validation MAE: {LEADER['DAYS'].sort_values('val_mae').iloc[0]['val_mae']:.3f}")
print(f"31. Best DAYS validation R²: {LEADER['DAYS'].sort_values('val_mae').iloc[0]['val_r2']:.4f}")
print(f"32. Best KM validation MAE: {LEADER['KM'].sort_values('val_mae').iloc[0]['val_mae']:.3f}")
print(f"33. Best KM validation R²: {LEADER['KM'].sort_values('val_mae').iloc[0]['val_r2']:.4f}")
print(f"34. Ensemble DAYS improvement: {ENS['DAYS']['best_single_val_mae']-ENS['DAYS']['blend_val_mae']:+.3f} MAE "
      f"(accepted={ENS['DAYS']['accepted']})")
print(f"35. Ensemble KM improvement: {ENS['KM']['best_single_val_mae']-ENS['KM']['blend_val_mae']:+.3f} MAE "
      f"(accepted={ENS['KM']['accepted']})")
print(f"36. Specialist model used?: DAYS={SPECIALIST['DAYS']['used']} KM={SPECIALIST['KM']['used']}")
print(f"37. Specialist model benefit: " +
      "; ".join(f"{t}: " + (f"routed VAL {SPECIALIST[t].get('routed_full_val_mae', float('nan')):.2f} vs single "
                            f"{SPECIALIST[t].get('single_full_val_mae', float('nan')):.2f}"
                            if 'routed_full_val_mae' in SPECIALIST[t] else SPECIALIST[t].get('reason', 'n/a'))
                for t in ('DAYS', 'KM')))
print(f"38. BASE TEST DAYS MAE: {tc.loc['DAYS','baseline_mae']:.3f}")
print(f"39. TUNED TEST DAYS MAE: {tc.loc['DAYS','tuned_mae']:.3f}")
print(f"40. BASE TEST DAYS R²: {tc.loc['DAYS','baseline_r2']:.4f}")
print(f"41. TUNED TEST DAYS R²: {tc.loc['DAYS','tuned_r2']:.4f}")
print(f"42. DAYS MAE improvement %: {tc.loc['DAYS','mae_improvement_pct']:+.2f}%")
print(f"43. DAYS R² gain: {tc.loc['DAYS','r2_gain']:+.4f}")
print(f"44. BASE TEST KM MAE: {tc.loc['KM','baseline_mae']:.3f}")
print(f"45. TUNED TEST KM MAE: {tc.loc['KM','tuned_mae']:.3f}")
print(f"46. BASE TEST KM R²: {tc.loc['KM','baseline_r2']:.4f}")
print(f"47. TUNED TEST KM R²: {tc.loc['KM','tuned_r2']:.4f}")
print(f"48. KM MAE improvement %: {tc.loc['KM','mae_improvement_pct']:+.2f}%")
print(f"49. KM R² gain: {tc.loc['KM','r2_gain']:+.4f}")
print(f"50. Final DAYS ±30 accuracy: {P('DAYS','within_30'):.3f}")
print(f"51. Final DAYS ±60 accuracy: {P('DAYS','within_60'):.3f}")
print(f"52. Final KM ±500 accuracy: {P('KM','within_500'):.3f}")
print(f"53. Final KM ±1000 accuracy: {P('KM','within_1000'):.3f}")
print(f"54. Final KM ±2000 accuracy: {P('KM','within_2000'):.3f}")
print(f"55. Top 10 DAYS features: {list(IMP['DAYS'].head(10).source)}")
print(f"56. Top 10 KM features: {list(IMP['KM'].head(10).source)}")
print(f"57. Worst DAYS segment: {WORST[WORST.target=='DAYS'].iloc[0].to_dict() if (WORST.target=='DAYS').any() else 'n/a'}")
print(f"58. Worst KM segment: {WORST[WORST.target=='KM'].iloc[0].to_dict() if (WORST.target=='KM').any() else 'n/a'}")
print(f"59. Train/Validation/Test generalization: "
      + "; ".join(f"{t} TRAIN {LEADER[t].set_index('model').loc[BEST_SINGLE[t],'train_mae']:.1f} / "
                  f"VAL {LEADER[t].set_index('model').loc[BEST_SINGLE[t],'val_mae']:.1f} / "
                  f"TEST {tc.loc[t,'tuned_mae']:.1f}" for t in ('DAYS', 'KM')))
print(f"60. Overfitting concern: " + "; ".join(f"{t}={bool(LEADER[t].set_index('model').loc[BEST_SINGLE[t],'overfit_flag'])}"
                                               for t in ('DAYS', 'KM')))
print(f"61. Temporal drift concern: KM target median TRAIN {piv.loc['TRAIN','target_KM']:.0f} -> "
      f"TEST {piv.loc['TEST','target_KM']:.0f}; DAYS {piv.loc['TRAIN','target_DAYS']:.0f} -> {piv.loc['TEST','target_DAYS']:.0f}")
print(f"62. Reproducibility: {REPRO}")
_nbp = ROOT / "notebooks" / "12_v1_final_hyperparameter_tuning.ipynb"
_ncells = len(json.loads(_nbp.read_text())["cells"]) if _nbp.exists() else "n/a"
print(f"63. Notebook cells: {_ncells}")
print(f"64. Notebook errors: 0 (this run completed)")
print(f"65. Generated artifacts: {len(REQUIRED)-len(missing)}/{len(REQUIRED)} required present; "
      f"{n_figs} figures; {n_optuna} optuna trial tables; missing={missing}")
print(f"66. README updated: {'12_v1_final_hyperparameter_tuning.ipynb' in rp.read_text()}")
print(f"67. Final tuning verdict: {FINAL_VERDICT}")
print(f"68. V1 final status: {V1_STATUS}")
print(f"69. V2 survival recommended?: {V2_RECOMMENDED}")
print(f"\nQA: {'ALL PASS' if QA_ALL else 'FAIL -> ' + str(qa_df[~qa_df['pass']].check.tolist())}")
print("=" * 78)
print("NB12 DONE")